# RT-IoT2022 — Intrusion detection on IoT network traffic
## Deliverable 1 — Dataset loading and column familiarization

**Group members (same order as in the file name):**

| # | Last name | First name |
|---|-----------|------------|
| 1 | Ben Abderrazak | Mohamed Rached |
| 2 | Hasnaoui | Yasmine |
| 3 | Smaili | Ahmed |
| 4 | Zammouri | Mohamed Chrif |

**Course:** Artificial Intelligence — Instructor: Faouzi MOUSSA

## 0. Project context

- **Problem:** detect malicious traffic in an IoT network. Each row represents a network flow described by packet and connection statistics.
- **Goal of this notebook (L1):** load the dataset, report its size, show a few examples, and briefly explain the main column families.
- **Target:** `Attack_type` is the original traffic category; `label` is a binary target derived from it (`0` = benign, `1` = attack).

Detailed exploration, statistical analysis, visualizations, and data cleaning are reserved for the separate V1 notebook to avoid redundancy.

## 1. Setup


In [ ]:
from pathlib import Path
import fnmatch

import pandas as pd

DATA_PATH = Path("../Data/raw/dataset.csv")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


## 2. Loading the data

The raw dataset is read from the local project folder (`../Data/raw/dataset.csv`); no download is performed. We add the binary `label` column using the same rule as our initial notebook: MQTT, ThingSpeak, Wipro bulb, and Alexa traffic are considered benign (`0`), while other attack categories are labeled `1`.

This section also reports the number of observations and columns and displays a small sample, as required for L1.

In [ ]:
df = pd.read_csv(DATA_PATH)

NORMAL_KEYS = ("mqtt", "thing", "wipro", "alexa")
is_normal = df["Attack_type"].str.lower().str.contains("|".join(NORMAL_KEYS), na=False)
df["label"] = (~is_normal).astype(int)

TARGETS = ["Attack_type", "label"]

n_rows, n_cols = df.shape
print(f"Observations (rows): {n_rows:,}")
print(f"Columns: {n_cols}")
print("Original target: Attack_type")
print("Added binary target: label (0 = benign, 1 = attack)")

display(df.head(5))

## 3. Meaning of the main columns

Each row represents one **network flow** (communication between an originator and a responder). Prefixes `fwd_` and `bwd_` indicate forward and backward directions, while `flow_` generally summarizes both directions. The table below groups the feature columns and gives their meaning. This is a short familiarization overview; detailed variable analysis is in V1.

In [ ]:
# (group, patterns matching column names, meaning) — first matching group wins
COLUMN_GROUPS = [
    ("Ports", ["id.orig_p", "id.resp_p"], "Source port and destination port."),
    ("Protocol / service", ["proto", "service"], "Transport protocol and detected application service."),
    ("Duration", ["flow_duration"], "Duration of the flow in seconds."),
    ("Packet counts", ["fwd_pkts_tot", "bwd_pkts_tot", "fwd_data_pkts_tot", "bwd_data_pkts_tot"], "Packet counts in each direction, including packets carrying data."),
    ("Rates and ratios", ["fwd_pkts_per_sec", "bwd_pkts_per_sec", "flow_pkts_per_sec", "down_up_ratio", "payload_bytes_per_second"], "Packet rates, direction ratio, and payload rate."),
    ("Header sizes", ["*_header_size_*"], "Packet header size statistics by direction."),
    ("TCP flags", ["*_flag_count"], "Counts of TCP flags such as SYN, FIN, RST, ACK, PSH, URG, CWR, and ECE."),
    ("Payload sizes", ["*pkts_payload.*"], "Packet payload size statistics per direction and overall."),
    ("Inter-arrival times", ["*_iat.*"], "Statistics about time gaps between consecutive packets."),
    ("Subflows", ["*_subflow_*"], "Average packets or bytes per subflow, per direction."),
    ("Bulk transfer", ["*_bulk_*"], "Bytes, packets, and rates for bulk transfers, per direction."),
    ("Active / idle", ["active.*", "idle.*"], "Statistics about active and idle periods."),
    ("TCP window", ["*_window_size"], "TCP window sizes announced at the beginning and end of the flow."),
    ("Targets", ["Attack_type", "label"], "Attack_type is the original traffic class; label is our binary target (0 = benign, 1 = attack)."),
]

rows, assigned = [], set()
for group, patterns, meaning in COLUMN_GROUPS:
    cols = [c for c in df.columns if c not in assigned and any(fnmatch.fnmatchcase(c, p) for p in patterns)]
    assigned.update(cols)
    if cols:
        rows.append({"Group": group, "N": len(cols), "Columns": ", ".join(cols), "Meaning": meaning})

col_doc = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 110):
    display(col_doc[["Group", "N", "Meaning"]])

undocumented = [c for c in df.columns if c not in assigned]
print(f"Documented columns: {len(assigned)}/{len(df.columns)} | not documented: {undocumented}")

## 4. Conclusion

This L1 notebook fulfills the dataset-introduction requirements: local loading, row/column count, sample rows, the binary `label` creation, and a brief explanation of the principal feature groups. The detailed exploration (distributions, missing/infinite values, duplicates, outliers, correlations) and cleaning are kept in the separate V1 notebook so the work is not duplicated.